In [ ]:

from data_loader import DataLoader
from data_synthesizer.pipeline import load_all_results
from data_synthesizer.privacy_sampling import get_epsilon

from evaluation_report.ressemblance_report import ResemblanceReport
from evaluation_report.utility_report import UtilityReport
from evaluation_report.privacy_report import PrivacyReport 
from evaluation_report.privacy_anonymeter_report import PrivacyAnonymeterReport

In [2]:
import sys
from pathlib import Path

# Add repo root so `import experiments` works in notebooks.
sys.path.append(str(Path.cwd().parent.parent))

from experiments.paths import RESULTS_DIR, DATA_DIR, BASELINE_DIR

## Loading Data

### Real Data

In [3]:
cat_list_credit_card = ['SEX', 'EDUCATION', 'MARRIAGE', 'PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6', 'default.payment.next.month']
num_list_credit_card = ['LIMIT_BAL', 'AGE', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4','BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3','PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6']
credit_qai_columns = ['LIMIT_BAL','SEX','EDUCATION','MARRIAGE','AGE']
credit_risk_column = ['PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6','BILL_AMT1','BILL_AMT2','BILL_AMT3','BILL_AMT4','BILL_AMT5','BILL_AMT6','PAY_AMT1','PAY_AMT2','PAY_AMT3','PAY_AMT4','PAY_AMT5','PAY_AMT6']

df_real_credit_card_train = DataLoader(f'{DATA_DIR}/credit_card_Train.csv').get_dataframe(cat_list_credit_card,  drop_identation=True)
df_real_credit_card_test = DataLoader(f'{DATA_DIR}/credit_card_Test.csv').get_dataframe(cat_list_credit_card,  drop_identation=True)
credit_real_dict ={'data' : df_real_credit_card_train, 'cat_list' : cat_list_credit_card, 'num_list' : num_list_credit_card}

In [4]:
cat_list_adult = ['workclass','education','marital-status','occupation','relationship','race','sex','native-country','income']
num_list_adult = ['age','fnlwgt','education-num','capital-gain','capital-loss','hours-per-week']
adult_qai_columns = ['education','education-num','marital-status','occupation','relationship','sex', 'native-country']
adult_risk_column = ['capital-gain','capital-loss','hours-per-week','income', 'race']
df_real_adult_train = DataLoader(f'{DATA_DIR}/adult_train.csv').get_dataframe(cat_list_adult)
df_real_adult_test = DataLoader(f'{DATA_DIR}/adult_test.csv').get_dataframe(cat_list_adult)

In [5]:
num_list_cardio = ['age', 'height', 'weight', 'ap_hi', 'ap_lo']
cat_list_cardio = ['gender','cholesterol', 'gluc', 'smoke', 'alco', 'active', 'cardio']

cardio_qai_columns = ['age','gender','height','weight']
cardio_risk_column = ['ap_lo','ap_hi','cholesterol','gluc','smoke','alco','active']


df_real_cardio_train = DataLoader(f'{DATA_DIR}/cardio_train.csv').get_dataframe(cat_list_cardio)
df_real_cardio_test = DataLoader(f'{DATA_DIR}/cardio_test.csv').get_dataframe(cat_list_cardio)

### Synth Data

In [8]:
credit_ctgan  = load_all_results(f'{BASELINE_DIR}/credit_ctgan_baseline')
credit_tvae  = load_all_results(f'{BASELINE_DIR}/credit_tvae_baseline')
adult_ctgan  = load_all_results(f'{BASELINE_DIR}/adult_ctgan_baseline')
adult_tvae  = load_all_results(f'{BASELINE_DIR}/adult_tvae_baseline')
cardio_ctgan  = load_all_results(f'{BASELINE_DIR}/cardio_ctgan_baseline')
cardio_tvae  = load_all_results(f'{BASELINE_DIR}/cardio_tvae_baseline')


In [57]:
anonymeter = {
    'Credit CTGAN' : credit_ctgan['privacy_anonymeter_results'],
    'Credit TVAE' : credit_tvae['privacy_anonymeter_results'],
    'Adult CTGAN' : adult_ctgan['privacy_anonymeter_results'],
    'Adult TVAE' : adult_tvae['privacy_anonymeter_results'],
    'CARDIO CTGAN' : cardio_ctgan['privacy_anonymeter_results'],
    'CARDIO TVAE' : cardio_tvae['privacy_anonymeter_results'],    
}
anonymeter


{'Credit CTGAN': {'run_0': {'singling_univariate': AnonymeterResults(attacks_numbers=2000, attacks_succeeded=245, privacy_risk_original=0.12322368534867299, privacy_risk_control=0.09666487632783903, privacy_risk_naive=0.02391443155315765, specific_privacy=0.029400837324767528, specific_privacy_ci=0),
   'singling_multivariate': AnonymeterResults(attacks_numbers=2000, attacks_succeeded=263, privacy_risk_original=0.13220643192314172, privacy_risk_control=0.02618954948838195, privacy_risk_naive=0.03389526108034512, specific_privacy=0.10886808862963103, specific_privacy_ci=0),
   'linkability_attacks': AnonymeterResults(attacks_numbers=2000, attacks_succeeded=2000, privacy_risk_original=0.010440311691454566, privacy_risk_control=0.00694702135693895, privacy_risk_naive=0.004950855451501456, specific_privacy=0.0035177280665216453, specific_privacy_ci=0)},
  'run_1': {'singling_univariate': AnonymeterResults(attacks_numbers=2000, attacks_succeeded=254, privacy_risk_original=0.1277150586359073

In [ ]:
credit_tvae_corrected  = load_all_results(f'{RESULTS_DIR}/mode_collapse_correction/credit_tvae_mode_collapse_corrected')
adult_tvae_corrected  = load_all_results(f'{RESULTS_DIR}/mode_collapse_correction/adult_tvae_mode_collapse_corrected')


In [30]:
credit_eval = ResemblanceReport(
    df_real_credit_card_train, 
    cat_list_credit_card,
    num_list_credit_card,
    {'CTGAN' : credit_ctgan, 'TVAE' : credit_tvae, 'TVAE-CORRECTED': credit_tvae_corrected})

credit_utility = UtilityReport(    df_real_credit_card_train, 
    cat_list_credit_card,
    num_list_credit_card,
    {'CTGAN' : credit_ctgan, 'TVAE' : credit_tvae, 'TVAE-CORRECTED': credit_tvae_corrected},
    )

adult_eval = ResemblanceReport(
    df_real_adult_train, 
    cat_list_adult,
    num_list_adult,
    {'CTGAN' : adult_ctgan, 'TVAE' : adult_tvae})

cardio_eval = ResemblanceReport(
    df_real_cardio_train, 
    cat_list_cardio,
    num_list_cardio,
    {'CTGAN' : cardio_ctgan, 'TVAE' : cardio_tvae})

In [43]:

credit_eval.get_categorical_univariate_report()

In [29]:
credit_eval.get_categorical_multivariate_report()

Accordion(children=(VBox(children=(Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x05\xdc\x00\x00\x…

In [34]:
credit_utility.get_report()

Accordion(children=(VBox(children=(VBox(children=(Label(value='Accuracy Means:'), HTML(value='<table border="1…

In [33]:
credit_tvae_corrected['utility_evaluation_results']

{'train_synthetic_test_real_results':                accuracy  precision    recall        f1  balanced_accuracy  \
 CART           0.727667   0.392354  0.445545  0.417261           0.626122   
 KNN            0.776333   0.487358  0.425743  0.454472           0.650145   
 LDA            0.818167   0.638404  0.389947  0.484161           0.664037   
 NB             0.486167   0.273077  0.811120  0.408594           0.603128   
 LR             0.815333   0.625767  0.388423  0.479323           0.661675   
 Random Forest  0.805500   0.583333  0.389185  0.466880           0.655655   
 SVM            0.760500   0.461202  0.561310  0.506355           0.688805   
 XGBoost        0.801667   0.568715  0.387662  0.461051           0.652653   
 
                precision_weighted  recall_weighted  f1_weighted  \
 CART                     0.740904         0.727667     0.733675   
 KNN                      0.766449         0.776333     0.770733   
 LDA                      0.800495         0.818167    

In [7]:
credit_tvae_heom_any_eps_01  = load_all_results(f'{RESULTS_DIR}/epsilon_comparison_heom_any/credit_tvae_eps_0.1')


In [35]:
# -*- coding: utf-8 -*-
"""
Coverage metrics for mode-collapse patching (per dataset × model).

Computes:
  - Columns with ZCR>0 (pre)
  - Total MAR (%)             [sum of real mass of missing levels pre-patch]
  - # Patched levels          [missing pre that reappear post]
  - SRR (post)                [support recovery rate across all categorical levels]
  - Δ mean JS (pp)            [mean per-column JS divergence (base-2) after - before, in percentage points]

Inputs:
  real_df, syn_before_df, syn_after_df : pandas.DataFrame
  categorical_cols : list[str]
  dataset_name, model_name: str labels for the output table

Author’s note: JS is divergence in base-2 (bits).
If you prefer JS *distance*, set return_distance=True in mean_js_by_column().
"""

from __future__ import annotations
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple

MISSING_TOKEN = "__MISSING__"

# ---------- Helpers

def _prep_cat(series: pd.Series) -> pd.Series:
    """Ensure categorical columns are strings and missing is a proper level."""
    return series.astype("object").fillna(MISSING_TOKEN).astype(str)

def _aligned_probs(real_col: pd.Series, syn_col: pd.Series) -> Tuple[pd.Index, np.ndarray, np.ndarray]:
    """
    Return (levels, p_real, p_syn) aligned on the union of levels seen in either series.
    Probabilities sum to 1 over the union; unseen levels get 0.
    """
    r = _prep_cat(real_col)
    s = _prep_cat(syn_col)
    levels = pd.Index(r.unique()).union(pd.Index(s.unique()))
    pr = r.value_counts(normalize=True).reindex(levels, fill_value=0.0).values
    ps = s.value_counts(normalize=True).reindex(levels, fill_value=0.0).values
    # Normalize defensively (in case of edge cases); keeps zero-mass when empty
    pr = pr / pr.sum() if pr.sum() > 0 else pr
    ps = ps / ps.sum() if ps.sum() > 0 else ps
    return levels, pr, ps

def _js_divergence_base2(p: np.ndarray, q: np.ndarray) -> float:
    """Jensen–Shannon divergence (base-2)."""
    # Avoid log(0) by masking 0s as usual in KL
    m = 0.5 * (p + q)
    def _kl(a, b):
        mask = (a > 0) & (b > 0)
        return float(np.sum(a[mask] * (np.log(a[mask]) - np.log(b[mask])) / np.log(2)))
    return 0.5 * _kl(p, m) + 0.5 * _kl(q, m)

def mean_js_by_column(real_df: pd.DataFrame,
                      syn_df: pd.DataFrame,
                      categorical_cols: List[str],
                      return_distance: bool=False) -> float:
    """
    Mean per-column JS divergence (base-2) across categorical_cols.
    If return_distance=True, return sqrt(JS) (the JS distance) averaged across columns instead.
    """
    vals = []
    for c in categorical_cols:
        _, pr, ps = _aligned_probs(real_df[c], syn_df[c])
        js = _js_divergence_base2(pr, ps)
        vals.append(np.sqrt(js) if return_distance else js)
    return float(np.mean(vals)) if vals else np.nan

# ---------- Detector & coverage metrics

def per_column_missing_levels(real_df: pd.DataFrame,
                              syn_df: pd.DataFrame,
                              categorical_cols: List[str]) -> Dict[str, List[str]]:
    """
    Return dict: column -> list of real levels (as strings) that have zero count in syn_df.
    (Algorithm 1 in the paper; per-column cross-tab detector.)
    """
    missing: Dict[str, List[str]] = {}
    for c in categorical_cols:
        r = _prep_cat(real_df[c])
        s = _prep_cat(syn_df[c])
        # Real-level set:
        levels_r = pd.Index(r.unique())
        syn_counts_on_real = s.value_counts().reindex(levels_r, fill_value=0)
        missing_levels = levels_r[syn_counts_on_real.values == 0].tolist()
        missing[c] = [str(v) for v in missing_levels]
    return missing

def aggregate_mar(real_df: pd.DataFrame,
                  categorical_cols: List[str],
                  missing_levels: Dict[str, List[str]]) -> float:
    """
    Total Mass-At-Risk (MAR): sum over categorical columns of the real-data probability
    of levels that are missing in synthetic (pre-patch).
    Returns MAR as a fraction in [0,1]. Multiply by 100 for %.
    """
    total = 0.0
    for c in categorical_cols:
        r = _prep_cat(real_df[c])
        if len(r) == 0:
            continue
        p_r = r.value_counts(normalize=True)
        for lvl in missing_levels.get(c, []):
            total += float(p_r.get(lvl, 0.0))
    return float(total)

def support_recovery_rate(real_df: pd.DataFrame,
                          syn_df: pd.DataFrame,
                          categorical_cols: List[str]) -> float:
    """
    SRR: fraction of *all* real category levels (across the listed columns)
    that appear with non-zero frequency in syn_df.
    """
    total_levels = 0
    present_levels = 0
    for c in categorical_cols:
        r = _prep_cat(real_df[c])
        s = _prep_cat(syn_df[c])
        levels_r = pd.Index(r.unique())
        total_levels += len(levels_r)
        syn_counts_on_real = s.value_counts().reindex(levels_r, fill_value=0)
        present_levels += int((syn_counts_on_real.values > 0).sum())
    return present_levels / total_levels if total_levels > 0 else np.nan

# ---------- Main one-row summary

def summarize_mode_patch(real_df: pd.DataFrame,
                          syn_before_df: pd.DataFrame,
                          syn_after_df: pd.DataFrame,
                          categorical_cols: List[str],
                          dataset_name: str,
                          model_name: str,
                          use_js_distance: bool=False) -> Tuple[pd.DataFrame, Dict[str, Dict[str, List[str]]]]:
    """
    Compute the coverage table row and return (row_df, details).

    details: {
       'missing_pre': {col: [levels...]},
       'still_missing_post': {col: [levels...]},
       'patched_levels': {col: [levels that were missing then recovered]}
    }
    """
    # 1) Missing levels pre and post
    missing_pre = per_column_missing_levels(real_df, syn_before_df, categorical_cols)
    missing_post = per_column_missing_levels(real_df, syn_after_df, categorical_cols)

    # 2) Counts for the table
    cols_with_zcr = sum(1 for c in categorical_cols if len(missing_pre.get(c, [])) > 0)
    mar = aggregate_mar(real_df, categorical_cols, missing_pre)  # fraction
    # Patched = missing pre that are now present
    patched = {
        c: sorted(list(set(missing_pre.get(c, [])) - set(missing_post.get(c, []))))
        for c in categorical_cols
    }
    n_patched_levels = sum(len(v) for v in patched.values())
    srr_post = support_recovery_rate(real_df, syn_after_df, categorical_cols)

    # 3) Mean per-column JS (or distance) before/after and delta (pp)
    js_func = (lambda R, S: mean_js_by_column(R, S, categorical_cols, return_distance=use_js_distance))
    mean_js_before = js_func(real_df, syn_before_df)
    mean_js_after  = js_func(real_df, syn_after_df)
    delta_js_pp = 100.0 * (mean_js_after - mean_js_before)

    row = pd.DataFrame([{
        "Dataset": dataset_name,
        "Model": model_name,
        "Cols with ZCR>0 (pre)": int(cols_with_zcr),
        "Total MAR (%)": round(100.0 * mar, 3),
        "# Patched levels": int(n_patched_levels),
        "SRR (post)": round(float(srr_post), 3) if srr_post == srr_post else np.nan,
        "Δ mean JS (pp)": round(delta_js_pp, 3),
        # Optional: export the raw means for completeness (comment out if not needed)
        "Mean JS (pre)": round(mean_js_before, 6),
        "Mean JS (post)": round(mean_js_after, 6),
    }])

    details = {
        "missing_pre": missing_pre,
        "still_missing_post": missing_post,
        "patched_levels": patched,
    }
    return row, details

# ---------- Example usage (edit paths and categorical lists to your setup)

    # Example: set your own file paths here (CSV or Parquet; replace with your paths)
    # CREDIT (TVAE)
    # real_credit = pd.read_csv("credit_real.csv")
    # syn_credit_before = pd.read_csv("credit_tvae_before.csv")
    # syn_credit_after  = pd.read_csv("credit_tvae_after_patch.csv")
    # credit_cats = ["SEX","EDUCATION","MARRIAGE","PAY_0","PAY_2","PAY_3","PAY_4","PAY_5","PAY_6",
    #                "default.payment.next.month"]  # adjust to your schema

    # ADULT (TVAE)
    # real_adult = pd.read_csv("adult_real.csv")
    # syn_adult_before = pd.read_csv("adult_tvae_before.csv")
    # syn_adult_after  = pd.read_csv("adult_tvae_after_patch.csv")
    # adult_cats = ["workclass","education","marital-status","occupation","relationship","race","sex","native-country","income"]

    # ---- run summaries (uncomment and adapt) ----
    # rows = []
    # r1, d1 = summarize_mode_patch(real_credit, syn_credit_before, syn_credit_after, credit_cats,
    #                               dataset_name="Credit", model_name="TVAE", use_js_distance=False)
    # rows.append(r1)
    # r2, d2 = summarize_mode_patch(real_adult, syn_adult_before, syn_adult_after, adult_cats,
    #                               dataset_name="Adult", model_name="TVAE", use_js_distance=False)
    # rows.append(r2)
    # results = pd.concat(rows, ignore_index=True)
    # print(results.to_string(index=False))
    # # Optional: inspect details (per-column lists)
    # # print(d1["missing_pre"]); print(d1["patched_levels"]); print(d1["still_missing_post"])
    # # print(d2["missing_pre"]); print(d2["patched_levels"]); print(d2["still_missing_post"])
    # # results.to_csv("mode_patch_coverage_table.csv", index=False)


In [37]:
syn_before_df = credit_tvae['generation_results']["synthetic_data"]
syn_after_df = credit_tvae_corrected['generation_results']["synthetic_data"]

row, details = summarize_mode_patch(
    df_real_credit_card_train, syn_before_df, syn_after_df,
    categorical_cols=cat_list_credit_card,
    dataset_name="Credit",   # or "Adult"
    model_name="TVAE",
    use_js_distance=False    # True if you prefer JS distance (sqrt(JS))
)

In [39]:
details

{'missing_pre': {'SEX': [],
  'EDUCATION': ['0'],
  'MARRIAGE': [],
  'PAY_0': [],
  'PAY_2': ['8'],
  'PAY_3': ['1'],
  'PAY_4': ['1'],
  'PAY_5': [],
  'PAY_6': [],
  'default.payment.next.month': []},
 'still_missing_post': {'SEX': [],
  'EDUCATION': [],
  'MARRIAGE': [],
  'PAY_0': [],
  'PAY_2': [],
  'PAY_3': [],
  'PAY_4': [],
  'PAY_5': [],
  'PAY_6': [],
  'default.payment.next.month': []},
 'patched_levels': {'SEX': [],
  'EDUCATION': ['0'],
  'MARRIAGE': [],
  'PAY_0': [],
  'PAY_2': ['8'],
  'PAY_3': ['1'],
  'PAY_4': ['1'],
  'PAY_5': [],
  'PAY_6': [],
  'default.payment.next.month': []}}

In [38]:
row

,Dataset,Model,Cols with ZCR>0 (pre),Total MAR (%),# Patched levels,SRR (post),Δ mean JS (pp),Mean JS (pre),Mean JS (post)
0,Credit,TVAE,4,0.058,4,1.0,0.003,0.008281,0.008309


In [41]:
syn_before_df = adult_tvae['generation_results']["synthetic_data"]
syn_after_df = adult_tvae_corrected['generation_results']["synthetic_data"]

row, details = summarize_mode_patch(
    df_real_adult_train, syn_before_df, syn_after_df,
    categorical_cols=cat_list_adult,
    dataset_name="Adult",   # or "Adult"
    model_name="TVAE",
    use_js_distance=False    # True if you prefer JS distance (sqrt(JS))
)

In [42]:
row


,Dataset,Model,Cols with ZCR>0 (pre),Total MAR (%),# Patched levels,SRR (post),Δ mean JS (pp),Mean JS (pre),Mean JS (post)
0,Adult,TVAE,1,0.003,1,1.0,0.001,0.004774,0.004784


In [44]:
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple

MISSING_SENTINEL = "<<MISSING>>"

# -------------------------------
# KL / JS helpers (base-2 logs)
# -------------------------------
def _safe_kl_base2(p: np.ndarray, q: np.ndarray) -> float:
    """KL(p || q) with base-2 logs; entries with p=0 contribute 0."""
    mask = p > 0
    return float(np.sum(p[mask] * (np.log(p[mask] / q[mask]) / np.log(2))))

def js_divergence_base2(p: np.ndarray, q: np.ndarray) -> float:
    """Jensen–Shannon divergence (base-2); in [0, 1]."""
    m = 0.5 * (p + q)
    return 0.5 * _safe_kl_base2(p, m) + 0.5 * _safe_kl_base2(q, m)

def js_distance_base2(p: np.ndarray, q: np.ndarray) -> float:
    """Jensen–Shannon distance (base-2): sqrt of divergence; in [0, 1]."""
    return float(np.sqrt(js_divergence_base2(p, q)))

# -------------------------------
# Categorical utilities
# -------------------------------
def _as_cat_series(s: pd.Series) -> pd.Series:
    """Cast to object, keep NaN as a proper level via sentinel."""
    return s.astype("object").where(s.notna(), MISSING_SENTINEL).astype(str)

def _level_probs(series: pd.Series, levels: pd.Index) -> pd.Series:
    """Return probability mass function over `levels` (sum=1)."""
    counts = _as_cat_series(series).value_counts(dropna=False)
    counts = counts.reindex(levels, fill_value=0)
    total = counts.sum()
    if total == 0:
        return pd.Series(np.zeros(len(levels), dtype=float), index=levels)
    return counts / total

def _missing_levels(real: pd.Series, synth: pd.Series, levels: pd.Index) -> List[str]:
    """Levels present in real (levels) with zero count in synth."""
    scounts = _as_cat_series(synth).value_counts(dropna=False)
    scounts = scounts.reindex(levels, fill_value=0)
    return [lvl for lvl in levels if scounts.loc[lvl] == 0]

# -------------------------------
# Per-column stats (with JSDist)
# -------------------------------
def column_mode_stats(
    real_df: pd.DataFrame,
    syn_pre_df: pd.DataFrame,
    syn_post_df: pd.DataFrame,
    col: str
) -> Dict:
    r = _as_cat_series(real_df[col])
    s0 = _as_cat_series(syn_pre_df[col])
    s1 = _as_cat_series(syn_post_df[col])

    # Level universe = REAL (per your detector)
    levels = pd.Index(r.unique())

    # Discrete pmfs
    pR = _level_probs(r, levels)
    p0 = _level_probs(s0, levels)
    p1 = _level_probs(s1, levels)

    # Missing levels pre / post
    miss_pre = _missing_levels(r, s0, levels)
    miss_post = _missing_levels(r, s1, levels)

    # MAR(c) = sum of real mass of missing levels (pre)
    mar_c = float(pR.loc[miss_pre].sum())  # in [0,1]

    # ZCR(c): fraction of levels missing in pre
    zcr_c = (len(miss_pre) / len(levels)) if len(levels) > 0 else 0.0

    # JS **distance** per column (base-2)
    jsd_pre = js_distance_base2(pR.to_numpy(), p0.to_numpy())
    jsd_post = js_distance_base2(pR.to_numpy(), p1.to_numpy())

    # How many of the previously missing levels re-appeared after patch?
    patched_levels = sum(1 for lvl in miss_pre if lvl not in miss_post)

    return dict(
        column=col,
        n_levels=len(levels),
        zcr_pre=zcr_c,
        mar_pre=mar_c,
        jsd_pre=jsd_pre,
        jsd_post=jsd_post,
        # If you want backward-compatibility with earlier names, uncomment:
        # js_pre=jsd_pre, js_post=jsd_post,
        miss_pre=miss_pre,
        miss_post=miss_post,
        patched_levels=patched_levels
    )

# -------------------------------
# Dataset-level summary (with JSDist)
# -------------------------------
def summarize_dataset(
    dataset_name: str,
    model_name: str,
    real_df: pd.DataFrame,
    syn_pre_df: pd.DataFrame,
    syn_post_df: pd.DataFrame,
    categorical_cols: List[str]
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns:
      - summary_row: 1-row DataFrame with the requested table metrics
        (now using JS *distance*)
      - details: per-column DataFrame with MAR/ZCR/JSDist and missing-level lists
    """
    details = []
    for c in categorical_cols:
        if c not in real_df.columns or c not in syn_pre_df.columns or c not in syn_post_df.columns:
            raise KeyError(f"Column '{c}' must exist in real, syn_pre and syn_post.")
        details.append(column_mode_stats(real_df, syn_pre_df, syn_post_df, c))
    details_df = pd.DataFrame(details)

    # Aggregations
    cols_with_zcr_any = int((details_df["zcr_pre"] > 0).sum())
    total_mar_percent = 100.0 * float(details_df["mar_pre"].sum())  # %
    total_patched_levels = int(details_df["patched_levels"].sum())

    # SRR(post): fraction of all real levels that are present post-patch
    total_levels = int(details_df["n_levels"].sum())
    missing_post_total = int(details_df["miss_post"].apply(len).sum())
    srr_post = float(1.0 - (missing_post_total / total_levels)) if total_levels > 0 else np.nan

    # Δ mean JSD (pp): (mean_post - mean_pre) * 100
    delta_mean_jsd_pp = 100.0 * float(details_df["jsd_post"].mean() - details_df["jsd_pre"].mean())

    summary_row = pd.DataFrame([{
        "Dataset": dataset_name,
        "Model": model_name,
        "Columns with ZCR>0 (pre)": cols_with_zcr_any,
        "Total MAR (%)": total_mar_percent,
        "# Patched levels": total_patched_levels,
        "SRR (post)": srr_post,
        "Δ mean JSD (pp)": delta_mean_jsd_pp
    }])

    return summary_row, details_df[[
        "column", "n_levels", "zcr_pre", "mar_pre", "jsd_pre", "jsd_post",
        "miss_pre", "miss_post", "patched_levels"
    ]]

# -------------------------------
# Multi-dataset convenience
# -------------------------------
def summarize_many(runs: Dict[str, Dict]) -> Tuple[pd.DataFrame, Dict[str, pd.DataFrame]]:
    """
    runs = {
       "Credit TVAE": {
           "model": "TVAE",
           "real": real_credit_df,
           "syn_pre": syn_credit_uncorrected_df,
           "syn_post": syn_credit_patched_df,
           "categoricals": cat_cols_credit
       },
       "Adult TVAE": {
           "model": "TVAE",
           "real": real_adult_df,
           "syn_pre": syn_adult_uncorrected_df,
           "syn_post": syn_adult_patched_df,
           "categoricals": cat_cols_adult
       },
    }
    """
    all_rows = []
    per_dataset_details = {}
    for name, cfg in runs.items():
        row, details = summarize_dataset(
            dataset_name=name,
            model_name=cfg.get("model", "TVAE"),
            real_df=cfg["real"],
            syn_pre_df=cfg["syn_pre"],
            syn_post_df=cfg["syn_post"],
            categorical_cols=cfg["categoricals"]
        )
        all_rows.append(row)
        per_dataset_details[name] = details
    return pd.concat(all_rows, ignore_index=True), per_dataset_details


In [45]:
runs = {
    "Credit TVAE": {
        "model": "TVAE",
        "real": df_real_credit_card_train,
        "syn_pre": credit_tvae['generation_results']["synthetic_data"],
        "syn_post": credit_tvae_corrected['generation_results']["synthetic_data"],
        "categoricals": cat_list_credit_card
    },
    "Adult TVAE": {
        "model": "TVAE",
        "real": df_real_adult_train,
        "syn_pre": adult_tvae['generation_results']["synthetic_data"],
        "syn_post": adult_tvae_corrected['generation_results']["synthetic_data"],
        "categoricals": cat_list_adult
    },
}

summary_table, details_by_dataset = summarize_many(runs)

In [46]:
summary_table

,Dataset,Model,Columns with ZCR>0 (pre),Total MAR (%),# Patched levels,SRR (post),Δ mean JSD (pp)
0,Credit TVAE,TVAE,4,0.058333,4,1.0,0.024499
1,Adult TVAE,TVAE,1,0.003071,1,1.0,0.009298


In [48]:
details_by_dataset['Credit TVAE']


,column,n_levels,zcr_pre,mar_pre,jsd_pre,jsd_post,miss_pre,miss_post,patched_levels
0,SEX,2,0.000000,0.000000,0.021786,0.022108,[],[],0
1,EDUCATION,7,0.142857,0.000417,0.156808,0.155810,[0],[],1
2,MARRIAGE,4,0.000000,0.000000,0.061381,0.061476,[],[],0
3,PAY_0,11,0.000000,0.000000,0.094788,0.094719,[],[],0
4,PAY_2,11,0.090909,0.000042,0.086968,0.087721,[8],[],1
5,PAY_3,11,0.090909,0.000083,0.085420,0.086678,[1],[],1
6,PAY_4,11,0.090909,0.000042,0.098620,0.099637,[1],[],1
7,PAY_5,10,0.000000,0.000000,0.091462,0.091580,[],[],0
8,PAY_6,10,0.000000,0.000000,0.089638,0.089904,[],[],0
9,default.payment.next.month,2,0.000000,0.000000,0.063299,0.062986,[],[],0
